# || Feature Engineering : Bureau Balance ||

In [135]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
print("All libraries imported!")
pd.set_option('display.float_format', lambda x: '%.3f' % x)

All libraries imported!


In [136]:
bureau_bal=pd.read_csv('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/raw/bureau_balance.csv')
bureau_bal

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C
...,...,...,...
27299920,5041336,-47,X
27299921,5041336,-48,X
27299922,5041336,-49,X
27299923,5041336,-50,X


In [137]:
# Ordinal Encoding the status of the loan in a month
categories={
    'C':0,
    'X':0,
    '0':0,
    '1':1,
    '2':2,
    '3':3,
    '4':4,
    '5':5
}
bureau_bal['STATUS']=bureau_bal['STATUS'].map(categories)

In [138]:
# a new df with unique loan ids
bb_fe=pd.DataFrame({
    'SK_ID_BUREAU': bureau_bal['SK_ID_BUREAU'].unique()
})
bb_fe

,SK_ID_BUREAU
0,5715448
1,5715449
2,5715451
3,5715452
4,5715453
...,...
817390,5041141
817391,5041143
817392,5041172
817393,5041332


#### The features that can be extracted from MONTHS_BALANCE for each loan are : 
1. bb_num_months : number of months for which history is available, 
2. bb_latest_mon : latest month for which status is available,
3. bb_oldest_mon : oldest month for which status is available, 
4. bb_history_age : total age of the loan history.

In [139]:
bb_fe['bb_oldest_mon']=bb_fe['SK_ID_BUREAU'].map(bureau_bal.groupby(
    'SK_ID_BUREAU'
)['MONTHS_BALANCE'].min())

bb_fe['bb_latest_mon']=bb_fe['SK_ID_BUREAU'].map(bureau_bal.groupby(
    'SK_ID_BUREAU'
)['MONTHS_BALANCE'].max())

bb_fe['bb_history_age']=bb_fe['SK_ID_BUREAU'].map(abs(bureau_bal.groupby(
    'SK_ID_BUREAU'
)['MONTHS_BALANCE'].min()))

In [140]:
bb_fe['bb_num_mon']=-bb_fe['bb_oldest_mon']+bb_fe['bb_latest_mon']+1

In [141]:
bb_fe

,SK_ID_BUREAU,bb_oldest_mon,bb_latest_mon,bb_history_age,bb_num_mon
0,5715448,-26,0,26,27
1,5715449,-11,0,11,12
2,5715451,-30,-5,30,26
3,5715452,-32,0,32,33
4,5715453,-37,0,37,38
...,...,...,...,...,...
817390,5041141,-24,0,24,25
817391,5041143,-96,-14,96,83
817392,5041172,-77,0,77,78
817393,5041332,-23,0,23,24


#### Now, the features that can be engineered from status :
Firstly, the status can be encoded as C,X to 0 and rest 1 to 5 because these are ordinal. Larger is worse for the credit score of the loan and hence borrower.
1. bb_max_default : the max DPD a loan has went into.
2. bb_curr_stat : the current status of the loan. This can be OHE because the trees are better at binary splitting.
Now, for a borrower that is having irregular payments frequently, max and current will not capture this irregularity.
3. bb_mean_dpd : the mean of status for all recorded months.
4. bb_sma0 : no of times the loan was 0-30 DPD.
5. bb_sma1 : no of times the account was irregular for 31+ Days.
6. bb_sma2 : no of times the account was irregular for 61+ Days.
7. bb_npa : no of times loan went npa (DPD>90).
Now, the ratio for these counts of overdues. This is done as ratio captures behaviour better than count. 2/10 and 2/100 are not same.
8. bb_sma0_ratio : bb_sma0/bb_num_mon
9. bb_sma1_ratio : bb_sma1/bb_num_mon
10. bb_sma2_ratio : bb_sma1/bb_num_mon
11. bb_npa_ratio : bb_npa/bb_num_mon

In [142]:
bb_fe['bb_max_default']=bb_fe['SK_ID_BUREAU'].map(bureau_bal.groupby(
    'SK_ID_BUREAU'
)['STATUS'].max())

bb_fe['bb_curr_stat'] = bb_fe['SK_ID_BUREAU'].map(
    bureau_bal.sort_values(
        ['SK_ID_BUREAU','MONTHS_BALANCE'],
        ascending=[True,False]
    )
    .groupby('SK_ID_BUREAU')['STATUS']
    .first()
)

In [143]:
bb_fe['bb_mean_dpd']=bb_fe['SK_ID_BUREAU'].map(bureau_bal.groupby(
    'SK_ID_BUREAU'
)['STATUS'].mean())

In [144]:
bb_fe['bb_sma0']=bb_fe['SK_ID_BUREAU'].map(bureau_bal[bureau_bal['STATUS']==1].groupby(
    'SK_ID_BUREAU'
).size())

bb_fe['bb_sma1']=bb_fe['SK_ID_BUREAU'].map(bureau_bal[bureau_bal['STATUS']==2].groupby(
    'SK_ID_BUREAU'
)['STATUS'].size())

bb_fe['bb_sma2']=bb_fe['SK_ID_BUREAU'].map(bureau_bal[bureau_bal['STATUS']==3].groupby(
    'SK_ID_BUREAU'
)['STATUS'].size())

bb_fe['bb_npa']=bb_fe['SK_ID_BUREAU'].map(bureau_bal[bureau_bal['STATUS']>=4].groupby(
    'SK_ID_BUREAU'
)['STATUS'].size())

cols = [
    'bb_sma0',
    'bb_sma1',
    'bb_sma2',
    'bb_npa'
]

bb_fe[cols] = bb_fe[cols].fillna(0)

In [145]:
bb_fe['bb_sma0_ratio']=bb_fe['bb_sma0']/bb_fe['bb_num_mon']
bb_fe['bb_sma1_ratio']=bb_fe['bb_sma1']/bb_fe['bb_num_mon']
bb_fe['bb_sma2_ratio']=bb_fe['bb_sma2']/bb_fe['bb_num_mon']
bb_fe['bb_npa_ratio']=bb_fe['bb_npa']/bb_fe['bb_num_mon']

In [146]:
bb_fe

,SK_ID_BUREAU,bb_oldest_mon,bb_latest_mon,bb_history_age,bb_num_mon,bb_max_default,bb_curr_stat,bb_mean_dpd,bb_sma0,bb_sma1,bb_sma2,bb_npa,bb_sma0_ratio,bb_sma1_ratio,bb_sma2_ratio,bb_npa_ratio
0,5715448,-26,0,26,27,0,0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,5715449,-11,0,11,12,0,0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2,5715451,-30,-5,30,26,0,0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
3,5715452,-32,0,32,33,0,0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
4,5715453,-37,0,37,38,0,0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
817390,5041141,-24,0,24,25,0,0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
817391,5041143,-96,-14,96,83,0,0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
817392,5041172,-77,0,77,78,0,0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
817393,5041332,-23,0,23,24,0,0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
